# Часть 5 из 8. Новые признаки и CatBoost на них

**Проект:** прогноз цены квартиры на вторичном рынке Москвы по датасету `Moscow_Real_Estate`.

**В этой части:**
- расстояния до Кремля, Москва-Сити, Лужников и ВДНХ;
- отношения площадей и взаимодействия признаков, средние по району и станции метро;
- CatBoost на 41 признаке и подбор гиперпараметров Optuna.

**Главный результат:** Optuna — MAPE 6,665 % на валидации; параметры сохранены в `best_catboost_params.json`.

> Код продолжает предыдущие части и использует их переменные. Результаты уже сохранены в блокноте, поэтому его можно читать без запуска. Чтобы пересчитать часть, сначала выполните в этом же ядре части 1–4 по порядку: `%run -i 01_eda.ipynb` и так далее.

← [Часть 4. Эксперименты с CatBoost](04_catboost_experiments.ipynb) · [Все части](README.md) · [Часть 6. Ансамбль CatBoost, LightGBM и XGBoost](06_ensemble.ipynb) →

### Feature engineering

In [57]:
# Расстояние до известных мест
# Координаты известных мест
landmarks = {
    'kremlin':   (37.6173, 55.7558),
    'msk_city':  (37.5370, 55.7495),
    'luzhniki':  (37.5544, 55.7158),
    'vdnkh':     (37.6325, 55.8262),
}


import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    """Работает и со скалярами, и с pandas Series через numpy."""
    R = 6371.0  # км
    
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

for name, (lon, lat) in landmarks.items():
    df_cb[f'dist_{name}'] = haversine(df_cb['lat'], df_cb['lon'], lat, lon)

In [58]:
# Отношения и взаимодействия
df_cb['area_per_room']     = df_cb['total_area'] / df_cb['rooms'].clip(lower=1)
df_cb['kitchen_share']     = df_cb['kitchen_area'] / df_cb['total_area'].clip(lower=1)
df_cb['living_share']      = df_cb['living_area'] / df_cb['total_area'].clip(lower=1)
df_cb['ceiling_per_area']  = df_cb['ceiling_height'] / df_cb['total_area'].clip(lower=1)
df_cb['age_per_floor']     = df_cb['building_age'] / df_cb['floor'].clip(lower=1)
df_cb['center_x_area']     = df_cb['to_center_km'] * df_cb['total_area']
df_cb['metro_x_area']      = df_cb['metro_distance_min'] * df_cb['total_area']

In [59]:
# Агрегаты fit только на train; для отсутствующих групп используется среднее train.
from sklearn.model_selection import train_test_split

idx_tr, idx_tmp = train_test_split(df_cb.index, test_size=0.3, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=42)

AGGREGATE_SPECS = {
    'district_avg_area': ('district', 'total_area'),
    'district_avg_age': ('district', 'building_age'),
    'district_avg_ceiling': ('district', 'ceiling_height'),
    'station_avg_area': ('metro_station', 'total_area'),
}

def add_train_group_aggregates(frame, train_indices):
    mappings = {}
    train_frame = frame.loc[train_indices]
    for feature, (group_col, value_col) in AGGREGATE_SPECS.items():
        group_means = train_frame.groupby(group_col)[value_col].mean()
        fallback = float(train_frame[value_col].mean())
        frame[feature] = frame[group_col].map(group_means).fillna(fallback)
        mappings[feature] = {
            'group_column': group_col,
            'value_column': value_col,
            'fallback': fallback,
            'by_group': {str(key): float(value) for key, value in group_means.items()},
        }
    return frame, mappings

df_cb, aggregate_mappings = add_train_group_aggregates(df_cb, idx_tr)

### CatBoost с новыми фичами

In [60]:
TARGET = 'price_rub'

# date-подобные
bad = [c for c in df_cb.columns
       if isinstance(df_cb[c].dtype, pd.PeriodDtype)
       or pd.api.types.is_datetime64_any_dtype(df_cb[c])]
df_cb = df_cb.drop(columns=bad)

# Категориальные — строки для CatBoost
cat_cols = [c for c in ['district', 'okrug', 'metro_station', 'metro_line',
                        'building_type', 'renovation']
            if c in df_cb.columns]
for c in cat_cols:
    df_cb[c] = df_cb[c].astype(str)

X = df_cb.drop(columns=[TARGET])
y = df_cb[TARGET].values
y_log = np.log1p(y)

# Один и тот же split для всех экспериментов
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y_log, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)

cat_idx = [X.columns.get_loc(c) for c in cat_cols]
print(f"train: {X_tr.shape}, val: {X_val.shape}, test: {X_test.shape}")

train: (35000, 41), val: (7500, 41), test: (7500, 41)


In [61]:
model_1 = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    early_stopping_rounds=200,
    verbose=500,
    cat_features=cat_idx,
)

t0 = time.time()
model_1.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
print(f"Обучение: {time.time() - t0:.1f} сек")

res_1 = evaluate_model(model_1, X_val, y_val, "Baseline с новыми фичами")

0:	learn: 0.0293358	test: 0.0290509	best: 0.0290509 (0)	total: 85.9ms	remaining: 7m 9s
500:	learn: 0.0042641	test: 0.0042695	best: 0.0042695 (500)	total: 42.7s	remaining: 6m 23s
1000:	learn: 0.0038871	test: 0.0041599	best: 0.0041598 (995)	total: 1m 26s	remaining: 5m 45s
1500:	learn: 0.0036346	test: 0.0041449	best: 0.0041448 (1498)	total: 2m 11s	remaining: 5m 6s
2000:	learn: 0.0034171	test: 0.0041401	best: 0.0041399 (1997)	total: 2m 55s	remaining: 4m 22s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.004139852652
bestIteration = 1997

Shrink model to first 1998 iterations.
Обучение: 195.2 сек

=== Baseline с новыми фичами ===
MAE:       880,621 руб
RMSE:    1,318,784 руб
MAPE:         6.73 %
R²:         0.9758


In [64]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Optuna с новыми фичами
def objective(trial):
    params = {
        'iterations': 10000,
        'learning_rate': trial.suggest_float('lr', 0.02, 0.05, log=True),   # узко
        'depth': trial.suggest_int('depth', 7, 9),                          # узко
        'l2_leaf_reg': trial.suggest_float('l2', 1, 6, log=True),           # узко
        'bagging_temperature': trial.suggest_float('bag', 0, 0.5),
        'random_strength': trial.suggest_float('rs', 0.1, 1.5, log=True),
        'border_count': 128,
        'loss_function': 'RMSE',
        'eval_metric': 'MAPE',
        'random_seed': 42,
        'early_stopping_rounds': 300,        # ← больше patience
        'verbose': False,
        'cat_features': cat_idx,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    pred = np.expm1(model.predict(X_val))
    true = np.expm1(y_val)
    return np.mean(np.abs((pred - true) / true)) * 100

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

  0%|          | 0/40 [00:00<?, ?it/s]

In [67]:
print("Best MAPE:", study.best_value)
print("Best trial:", study.best_trial.number)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Best MAPE: 6.665222130690503
Best trial: 13
Best params:
  lr: 0.02903980827749969
  depth: 7
  l2: 4.142779709085563
  bag: 0.10470460639408721
  rs: 0.1987792994929149


In [68]:
import json

best_params = study.best_params.copy()
best_params.update({
    'iterations': 10000,
    'loss_function': 'RMSE',
    'eval_metric': 'MAPE',
    'random_seed': 42,
    'early_stopping_rounds': 300,
    'verbose': 500,
    'cat_features': cat_idx,
})

with open('best_catboost_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print("Сохранено")

Сохранено


---
← [Часть 4. Эксперименты с CatBoost](04_catboost_experiments.ipynb) · [Все части](README.md) · [Часть 6. Ансамбль CatBoost, LightGBM и XGBoost](06_ensemble.ipynb) →